#  Topic Modeling and Trend Detection in Large Text Corpora

Topic modeling is a technique in natural language processing (NLP) and machine learning that aims to uncover latent thematic structures within a collection of texts. Topic modelling is a system learning technique that robotically discovers the principle themes or "topics" representing a huge document collection. The intention of topic modelling is to discover the hidden semantic systems within textual content facts, permitting customers to arrange, apprehend, and summarize the data in a manner that is each green and insightful.




## Arxiv Dataset
For this project we will use the Arxiv dataset which is a mirror of the original ArXiv data. For nearly 30 years, ArXiv has served the public and research communities by providing open access to scholarly articles, from the vast branches of physics to the many subdisciplines of computer science to everything in between, including math, statistics, electrical engineering, quantitative biology, and economics. This rich corpus of information offers significant, but sometimes overwhelming depth. Since the full arXiv dataset is quite large (approximately 1.1 TB and continuously growing), for the purposes of this project I will be using only the first 40,000 rows to reduce resource usage and ensure faster processing during development and experimentation.

### Install Bertopic

In [1]:
!pip install bertopic

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Import the necessary libraries

In [3]:
from umap import UMAP
import pandas as pd
from hdbscan import HDBSCAN
from sklearn.cluster import KMeans
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
import numpy as np
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from bertopic.dimensionality import BaseDimensionalityReduction
from bertopic.vectorizers import ClassTfidfTransformer
from sklearn.metrics import silhouette_score,calinski_harabasz_score,classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

### Fetch dataset

In [4]:
dataset = pd.read_csv('/content/drive/MyDrive/arxiv-40k.csv')

In [5]:
dataset

,update_date,title,journal-ref,submitter,authors,doi,comments,categories,abstract,id,license,report-no,versions,authors_parsed
0,2008-11-26,Calculation of prompt diphoton production cros...,"Phys.Rev.D76:013009,2007",Pavel Nadolsky,"C. Bal\'azs, E. L. Berger, P. M. Nadolsky, C.-...",10.1103/PhysRevD.76.013009,"37 pages, 15 figures; published version",hep-ph,A fully differential calculation in perturba...,704.0001,NaN,ANL-HEP-PR-07-12,"[{'version': 'v1', 'created': 'Mon, 2 Apr 2007...","[['Balázs', 'C.', ''], ['Berger', 'E. L.', '']..."
1,2008-12-13,Sparsity-certifying Graph Decompositions,NaN,Louis Theran,Ileana Streinu and Louis Theran,NaN,To appear in Graphs and Combinatorics,math.CO cs.CG,"We describe a new algorithm, the $(k,\ell)$-...",704.0002,http://arxiv.org/licenses/nonexclusive-distrib...,NaN,"[{'version': 'v1', 'created': 'Sat, 31 Mar 200...","[['Streinu', 'Ileana', ''], ['Theran', 'Louis'..."
2,2008-01-13,The evolution of the Earth-Moon system based o...,NaN,Hongjun Pan,Hongjun Pan,NaN,"23 pages, 3 figures",physics.gen-ph,The evolution of Earth-Moon system is descri...,704.0003,NaN,NaN,"[{'version': 'v1', 'created': 'Sun, 1 Apr 2007...","[['Pan', 'Hongjun', '']]"
3,2007-05-23,A determinant of Stirling cycle numbers counts...,NaN,David Callan,David Callan,NaN,11 pages,math.CO,We show that a determinant of Stirling cycle...,704.0004,NaN,NaN,"[{'version': 'v1', 'created': 'Sat, 31 Mar 200...","[['Callan', 'David', '']]"
4,2013-10-15,From dyadic $\Lambda_{\alpha}$ to $\Lambda_{\a...,"Illinois J. Math. 52 (2008) no.2, 681-689",Alberto Torchinsky,Wael Abu-Shammala and Alberto Torchinsky,NaN,NaN,math.CA math.FA,In this paper we show how to compute the $\L...,704.0005,NaN,NaN,"[{'version': 'v1', 'created': 'Mon, 2 Apr 2007...","[['Abu-Shammala', 'Wael', ''], ['Torchinsky', ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39995,2008-11-26,Flavorful Supersymmetry,"Phys.Rev.D77:075006,2008",Yasunori Nomura,"Yasunori Nomura, Michele Papucci, Daniel Stola...",10.1103/PhysRevD.77.075006,"20 pages; typos corrected, comments added, to ...",hep-ph,Weak scale supersymmetry provides elegant so...,712.2074,NaN,UCB-PTH-07/25,"[{'version': 'v1', 'created': 'Thu, 13 Dec 200...","[['Nomura', 'Yasunori', ''], ['Papucci', 'Mich..."
39996,2011-07-20,"($\ell,0)$-Carter partitions, a generating fun...","Electronic Journal of Combinatorics, Volume 15...",Chris Berg,"Chris Berg, Monica Vazirani",NaN,NaN,math.CO math.RT,In this paper we give an alternate combinato...,712.2075,http://arxiv.org/licenses/nonexclusive-distrib...,NaN,"[{'version': 'v1', 'created': 'Thu, 13 Dec 200...","[['Berg', 'Chris', ''], ['Vazirani', 'Monica',..."
39997,2007-12-20,On the irreducible representations of a finite...,NaN,Benjamin Steinberg,"Olexandr Ganyushkin, Volodymyr Mazorchuk and B...",NaN,NaN,math.RT math.GR,"Work of Clifford, Munn and Ponizovski{\u\i} ...",712.2076,NaN,NaN,"[{'version': 'v1', 'created': 'Thu, 13 Dec 200...","[['Ganyushkin', 'Olexandr', ''], ['Mazorchuk',..."
39998,2009-11-13,The Formation of Constellation III in the Larg...,NaN,Jason Harris,Jason Harris and Dennis Zaritsky,10.1071/AS07037,Accepted for publication in the Publications o...,astro-ph,We present a detailed reconstruction of the ...,712.2077,NaN,NaN,"[{'version': 'v1', 'created': 'Thu, 13 Dec 200...","[['Harris', 'Jason', ''], ['Zaritsky', 'Dennis..."


### Data Preprocessing

In [6]:
dataset.isnull().sum()

,0
update_date,0
title,0
journal-ref,19270
submitter,0
authors,0
doi,15629
comments,4709
categories,0
abstract,0
id,0


In [7]:
dataset = dataset.drop(columns=['update_date','title','journal-ref','submitter','authors','doi','comments','id','license','report-no','versions','authors_parsed'],axis=1)
dataset

,categories,abstract
0,hep-ph,A fully differential calculation in perturba...
1,math.CO cs.CG,"We describe a new algorithm, the $(k,\ell)$-..."
2,physics.gen-ph,The evolution of Earth-Moon system is descri...
3,math.CO,We show that a determinant of Stirling cycle...
4,math.CA math.FA,In this paper we show how to compute the $\L...
...,...,...
39995,hep-ph,Weak scale supersymmetry provides elegant so...
39996,math.CO math.RT,In this paper we give an alternate combinato...
39997,math.RT math.GR,"Work of Clifford, Munn and Ponizovski{\u\i} ..."
39998,astro-ph,We present a detailed reconstruction of the ...


In [8]:
threshold = 334
val_counts = dataset['categories'].value_counts()
rare_categories = val_counts[val_counts < threshold].index
dataset['categories'] = dataset['categories'].apply(
    lambda x: 'other' if x in rare_categories else x
)

In [9]:
dataset['categories'].value_counts()

,count
categories,
other,20024
astro-ph,6720
hep-ph,2222
quant-ph,1723
hep-th,1525
cond-mat.mtrl-sci,843
gr-qc,804
cond-mat.mes-hall,697
hep-ex,632


In [10]:
len(dataset['categories'].value_counts())

20

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    dataset['abstract'], dataset['categories'], test_size=0.2,
    stratify=dataset['categories'], random_state=42
)

## Create the BertTopic unsupervised Model 1

In [67]:
embedding_model = SentenceTransformer("all-MiniLM-L12-v2")

# Step 2 - Reduce dimensionality
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine')

# Step 3 - Cluster reduced embeddings
hdbscan_model = HDBSCAN(min_cluster_size=15,allow_single_cluster=True,metric='euclidean', algorithm='best', cluster_selection_method='eom', prediction_data=True)

# Step 4 - Tokenize topics
vectorizer_model = CountVectorizer(stop_words="english")

# Step 5 - Create topic representation
ctfidf_model = ClassTfidfTransformer()

# Step 6 - (Optional) Fine-tune topic representations with
# a `bertopic.representation` model
representation_model = KeyBERTInspired()

# All steps together
topic_model = BERTopic(
  embedding_model=embedding_model,          # Step 1 - Extract embeddings
  umap_model=umap_model,                    # Step 2 - Reduce dimensionality
  hdbscan_model=hdbscan_model,              # Step 3 - Cluster reduced embeddings
  vectorizer_model=vectorizer_model,        # Step 4 - Tokenize topics
  ctfidf_model=ctfidf_model,                # Step 5 - Extract topic words
  representation_model=representation_model # Step 6 - (Optional) Fine-tune topic representations
)

In [68]:
topics,probs = topic_model.fit_transform(X_train.tolist())

In [69]:
from collections import defaultdict, Counter

# Step 1: Map each topic to the most common category
topic_to_categories = defaultdict(list)
for topic, label in zip(topics, y_train):
    topic_to_categories[topic].append(label)

topic_to_main_category = {
    topic: Counter(labels).most_common(1)[0][0]
    for topic, labels in topic_to_categories.items()
}

# Step 2: Prepare training data for classifier
topic_keyword_texts = []
topic_categories = []

for topic_id in topic_to_main_category.keys():
    if topic_id == -1: continue  # skip outlier topic
    words = [word for word, _ in topic_model.get_topic(topic_id)[:10]]
    keyword_str = ' '.join(words)
    topic_keyword_texts.append(keyword_str)
    topic_categories.append(topic_to_main_category[topic_id])


### Evaluation on the BertTopic model

In [70]:
embeddings = topic_model.umap_model.embedding_
mask = np.array(topics) != -1
filtered_topics = np.array(topics)[mask]

print(f"Silhouette: {silhouette_score(embeddings[mask], filtered_topics):.3f}")
print(f"CH Index: {calinski_harabasz_score(embeddings[mask], filtered_topics):.0f}")

Silhouette: 0.571
CH Index: 44267


### The most frequent topics

In [71]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,14273,-1_galaxies_theory_dynamics_galaxy,"[galaxies, theory, dynamics, galaxy, quantum, ...",[ The renormalized mean value of the correspo...
1,0,376,0_decoding_channels_channel_decoder,"[decoding, channels, channel, decoder, codes, ...",[ We consider a state-dependent full-duplex r...
2,1,344,1_stochastic_diffusions_martingale_diffusion,"[stochastic, diffusions, martingale, diffusion...",[ A physical-mathematical approach to anomalo...
3,2,339,2_curvatures_curvature_ricci_riemannian,"[curvatures, curvature, ricci, riemannian, geo...",[ Along a Ricci flow solution on a closed man...
4,3,337,3_supersymmetric_supersymmetry_branes_strings,"[supersymmetric, supersymmetry, branes, string...",[ Hypermultiplet couplings in type IIA string...
...,...,...,...,...,...
238,237,16,237_polarizabilities_nucleon_neutron_polarisab...,"[polarizabilities, nucleon, neutron, polarisab...",[ Chiral Effective Field Theory is for photon...
239,238,16,238_tunneling_tunnel_tunnelling_barrier,"[tunneling, tunnel, tunnelling, barrier, quant...",[ Tunneling is an important physical process....
240,239,16,239_poisson_symplectic_quantizations_algebras,"[poisson, symplectic, quantizations, algebras,...",[ We construct quadratic finite-dimensional P...
241,240,15,240_cooperation_prisoner_evolutionary_altruism,"[cooperation, prisoner, evolutionary, altruism...",[ We study the problem of the emergence of co...


### Frequent words from the second most frequent topic

In [72]:
topic_model.get_topic(0)

[('decoding', np.float32(0.5264918)),
 ('channels', np.float32(0.49075168)),
 ('channel', np.float32(0.4745988)),
 ('decoder', np.float32(0.4714487)),
 ('codes', np.float32(0.47068015)),
 ('transmit', np.float32(0.46438283)),
 ('coded', np.float32(0.44723067)),
 ('multiplexing', np.float32(0.44347012)),
 ('coding', np.float32(0.42622015)),
 ('decode', np.float32(0.4248481))]

### Info about the documents clustered in these topics

In [73]:
topic_model.get_document_info(X_train)

,Document,Topic,Name,Representation,Representative_Docs,Top_n_words,Probability,Representative_document
0,A detailed comparison of the expressions for...,-1,-1_galaxies_theory_dynamics_galaxy,"[galaxies, theory, dynamics, galaxy, quantum, ...",[ The renormalized mean value of the correspo...,galaxies - theory - dynamics - galaxy - quantu...,0.000000,False
1,This is an expanded version of our earlier p...,-1,-1_galaxies_theory_dynamics_galaxy,"[galaxies, theory, dynamics, galaxy, quantum, ...",[ The renormalized mean value of the correspo...,galaxies - theory - dynamics - galaxy - quantu...,0.000000,False
2,Solutions to conservation laws satisfy the m...,-1,-1_galaxies_theory_dynamics_galaxy,"[galaxies, theory, dynamics, galaxy, quantum, ...",[ The renormalized mean value of the correspo...,galaxies - theory - dynamics - galaxy - quantu...,0.000000,False
3,By means of the 10-resonance unitary and ana...,237,237_polarizabilities_nucleon_neutron_polarisab...,"[polarizabilities, nucleon, neutron, polarisab...",[ Chiral Effective Field Theory is for photon...,polarizabilities - nucleon - neutron - polaris...,1.000000,False
4,A pedigree is a directed graph that describe...,14,14_genes_microarray_genome_gene,"[genes, microarray, genome, gene, transcriptio...",[ In order to express specific genes at the r...,genes - microarray - genome - gene - transcrip...,1.000000,False
...,...,...,...,...,...,...,...,...
31995,The high pressure phase diagram of CsC8 grap...,-1,-1_galaxies_theory_dynamics_galaxy,"[galaxies, theory, dynamics, galaxy, quantum, ...",[ The renormalized mean value of the correspo...,galaxies - theory - dynamics - galaxy - quantu...,0.000000,False
31996,We present a preliminary set of updated NLO ...,97,97_partons_parton_quarks_quark,"[partons, parton, quarks, quark, nucleons, had...",[ Nuclear parton distribution functions (NPDF...,partons - parton - quarks - quark - nucleons -...,0.784290,False
31997,We have investigated the doping dependence o...,18,18_superconductivity_superconductors_supercond...,"[superconductivity, superconductors, supercond...",[ Pairing of electrons in conventional superc...,superconductivity - superconductors - supercon...,0.707218,False
31998,We calculate the O($\eps$)--term of the two-...,-1,-1_galaxies_theory_dynamics_galaxy,"[galaxies, theory, dynamics, galaxy, quantum, ...",[ The renormalized mean value of the correspo...,galaxies - theory - dynamics - galaxy - quantu...,0.000000,False


## Data Visualization of the topics

In [74]:
topic_model.visualize_barchart(top_n_topics=20,n_words=10)

In [75]:
topic_model.visualize_heatmap()

In [76]:
topic_model.visualize_topics()

In [77]:
topic_model.visualize_hierarchy()

In [78]:
topics_per_class = topic_model.topics_per_class(X_train, classes=y_train)
topic_model.visualize_topics_per_class(topics_per_class)

In [79]:
vectorizer = TfidfVectorizer()
X_keywords = vectorizer.fit_transform(topic_keyword_texts)

clf = RandomForestClassifier(n_estimators=200,random_state=42)
clf.fit(X_keywords, topic_categories)

RandomForestClassifier(n_estimators=200, random_state=42)

In [80]:
test_topics, _ = topic_model.transform(X_test.tolist())

predicted_categories = []

for topic_id in test_topics:
    if topic_id == -1:
        predicted_categories.append("Unknown")
        continue
    words = [word for word, _ in topic_model.get_topic(topic_id)[:10]]
    keyword_str = ' '.join(words)
    X_input = vectorizer.transform([keyword_str])
    prediction = clf.predict(X_input)[0]
    predicted_categories.append(prediction)

In [81]:
print(classification_report(y_test, predicted_categories))

                    precision    recall  f1-score   support

           Unknown       0.00      0.00      0.00         0
          astro-ph       0.94      0.44      0.60      1344
 cond-mat.mes-hall       0.47      0.22      0.30       139
 cond-mat.mtrl-sci       0.45      0.09      0.15       169
    cond-mat.other       0.00      0.00      0.00        90
cond-mat.stat-mech       0.49      0.22      0.30       106
   cond-mat.str-el       0.59      0.26      0.36       115
     cs.IT math.IT       0.61      0.88      0.72        67
             gr-qc       0.41      0.15      0.22       161
            hep-ex       0.76      0.25      0.37       126
           hep-lat       0.76      0.40      0.52        70
            hep-ph       0.74      0.34      0.47       444
            hep-th       0.63      0.30      0.40       305
   math-ph math.MP       0.00      0.00      0.00        93
           math.AG       0.66      0.51      0.57        79
           math.CO       0.47      0.49

## Create BertTopic Model unsupervised 2

In [82]:
embedding_model = SentenceTransformer("all-mpnet-base-v2")


# Step 3 - Cluster reduced embeddings
hdbscan_model = HDBSCAN(min_cluster_size=15,allow_single_cluster=True,metric='euclidean', algorithm='best', cluster_selection_method='eom', prediction_data=True)

# Step 4 - Tokenize topics
vectorizer_model = TfidfVectorizer(stop_words="english")

# Step 5 - Create topic representation
ctfidf_model = ClassTfidfTransformer()
# Step 6 - (Optional) Fine-tune topic representations with
# a `bertopic.representation` model
representation_model = KeyBERTInspired()

# All steps together
topic_model_2 = BERTopic(
  embedding_model=embedding_model,          # Step 1 - Extract embeddings
  hdbscan_model=hdbscan_model,              # Step 3 - Cluster reduced embeddings
  vectorizer_model=vectorizer_model,        # Step 4 - Tokenize topics
  ctfidf_model = ctfidf_model,
  representation_model=representation_model # Step 6 - (Optional) Fine-tune topic representations
)

In [83]:
topics, _ = topic_model_2.fit_transform(X_train.tolist())

In [84]:
# Step 1: Map each topic to the most common category
topic_to_categories = defaultdict(list)
for topic, label in zip(topics, y_train):
    topic_to_categories[topic].append(label)

topic_to_main_category = {
    topic: Counter(labels).most_common(1)[0][0]
    for topic, labels in topic_to_categories.items()
}

# Step 2: Prepare training data for classifier
topic_keyword_texts = []
topic_categories = []

for topic_id in topic_to_main_category.keys():
    if topic_id == -1: continue  # skip outlier topic
    words = [word for word, _ in topic_model_2.get_topic(topic_id)[:10]]
    keyword_str = ' '.join(words)
    topic_keyword_texts.append(keyword_str)
    topic_categories.append(topic_to_main_category[topic_id])

### Evaluation

In [85]:
embeddings = topic_model_2.umap_model.embedding_
mask = np.array(topics) != -1
filtered_topics = np.array(topics)[mask]

print(f"Silhouette: {silhouette_score(embeddings[mask], filtered_topics):.3f}")
print(f"CH Index: {calinski_harabasz_score(embeddings[mask], filtered_topics):.0f}")

Silhouette: 0.566
CH Index: 7016


### The most frequent topics

In [86]:
topic_model_2.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,10959,-1_lhc_quark_higgs_supersymmetric,"[lhc, quark, higgs, supersymmetric, qcd, parti...",[ A novel mechanism of baryogenesis is propos...
1,0,5279,0_galaxies_redshifts_nebulae_galaxy,"[galaxies, redshifts, nebulae, galaxy, galacti...",[ The results obtained from a study of the ma...
2,1,892,1_colloids_colloidal_colloid_rheology,"[colloids, colloidal, colloid, rheology, molec...",[ In this article we calculate the surface ph...
3,2,640,2_superfluids_superfluidity_superfluid_fermionic,"[superfluids, superfluidity, superfluid, fermi...",[ We review and characterize the quantum cohe...
4,3,572,3_estimators_estimator_estimating_lasso,"[estimators, estimator, estimating, lasso, est...",[ This paper studies oracle properties of $\e...
...,...,...,...,...,...
203,202,16,202_polarons_phonons_phonon_polaronic,"[polarons, phonons, phonon, polaronic, polaron...",[ We investigate numerically various properti...
204,203,16,203_antiferromagnetic_electrons_fermi_fermion,"[antiferromagnetic, electrons, fermi, fermion,...",[ Comprehensive magnetic-field-orientation de...
205,204,16,204_heuristics_metaheuristic_heuristic_evoluti...,"[heuristics, metaheuristic, heuristic, evoluti...",[ We proposed a new search heuristic using th...
206,205,16,205_seismic_earthquakes_seismicity_earthquake,"[seismic, earthquakes, seismicity, earthquake,...",[ Two different way of assessing seismic vuln...


### Frequent words from the second most frequent topic

In [87]:
topic_model_2.get_topic(0)

[('galaxies', np.float32(0.5514057)),
 ('redshifts', np.float32(0.45929757)),
 ('nebulae', np.float32(0.45259702)),
 ('galaxy', np.float32(0.4524458)),
 ('galactic', np.float32(0.41739568)),
 ('nebula', np.float32(0.41042772)),
 ('quasars', np.float32(0.4035761)),
 ('ngc', np.float32(0.39951724)),
 ('redshift', np.float32(0.35261798)),
 ('stellar', np.float32(0.3445145))]

### Info about the documents clustered in these topics

In [88]:
topic_model_2.get_document_info(X_train)

,Document,Topic,Name,Representation,Representative_Docs,Top_n_words,Probability,Representative_document
0,A detailed comparison of the expressions for...,-1,-1_lhc_quark_higgs_supersymmetric,"[lhc, quark, higgs, supersymmetric, qcd, parti...",[ A novel mechanism of baryogenesis is propos...,lhc - quark - higgs - supersymmetric - qcd - p...,0.000000,False
1,This is an expanded version of our earlier p...,33,33_primes_primality_conjecture_integers,"[primes, primality, conjecture, integers, prim...",[ A classical problem in analytic number theo...,primes - primality - conjecture - integers - p...,0.379349,False
2,Solutions to conservation laws satisfy the m...,-1,-1_lhc_quark_higgs_supersymmetric,"[lhc, quark, higgs, supersymmetric, qcd, parti...",[ A novel mechanism of baryogenesis is propos...,lhc - quark - higgs - supersymmetric - qcd - p...,0.000000,False
3,By means of the 10-resonance unitary and ana...,21,21_nucleon_nuclei_antinucleon_protons,"[nucleon, nuclei, antinucleon, protons, neutri...",[ As a first step to analyze the electromagne...,nucleon - nuclei - antinucleon - protons - neu...,1.000000,False
4,A pedigree is a directed graph that describe...,-1,-1_lhc_quark_higgs_supersymmetric,"[lhc, quark, higgs, supersymmetric, qcd, parti...",[ A novel mechanism of baryogenesis is propos...,lhc - quark - higgs - supersymmetric - qcd - p...,0.000000,False
...,...,...,...,...,...,...,...,...
31995,The high pressure phase diagram of CsC8 grap...,9,9_graphene_graphite_graphitic_electron,"[graphene, graphite, graphitic, electron, elec...",[ We report the existence of zero energy surf...,graphene - graphite - graphitic - electron - e...,0.253120,False
31996,We present a preliminary set of updated NLO ...,-1,-1_lhc_quark_higgs_supersymmetric,"[lhc, quark, higgs, supersymmetric, qcd, parti...",[ A novel mechanism of baryogenesis is propos...,lhc - quark - higgs - supersymmetric - qcd - p...,0.000000,False
31997,We have investigated the doping dependence o...,4,4_superconductivity_superconductors_supercondu...,"[superconductivity, superconductors, supercond...",[ We analyze the ground state properties of a...,superconductivity - superconductors - supercon...,1.000000,False
31998,We calculate the O($\eps$)--term of the two-...,-1,-1_lhc_quark_higgs_supersymmetric,"[lhc, quark, higgs, supersymmetric, qcd, parti...",[ A novel mechanism of baryogenesis is propos...,lhc - quark - higgs - supersymmetric - qcd - p...,0.000000,False


## Visualization of the topics

In [89]:
topic_model_2.visualize_barchart(top_n_topics=20,n_words=8)

In [90]:
topic_model_2.visualize_heatmap()

In [91]:
topic_model_2.visualize_topics()

In [92]:
topic_model_2.visualize_hierarchy()

In [93]:
topics_per_class = topic_model_2.topics_per_class(X_train, classes=y_train)
topic_model_2.visualize_topics_per_class(topics_per_class)

In [94]:
vectorizer = TfidfVectorizer()
X_keywords = vectorizer.fit_transform(topic_keyword_texts)

clf = RandomForestClassifier(n_estimators=200,random_state=42)
clf.fit(X_keywords, topic_categories)

RandomForestClassifier(n_estimators=200, random_state=42)

In [96]:
test_topics, _ = topic_model_2.transform(X_test.tolist())

predicted_categories = []

for topic_id in test_topics:
    if topic_id == -1:
        predicted_categories.append("Unknown")
        continue
    words = [word for word, _ in topic_model.get_topic(topic_id)[:10]]
    keyword_str = ' '.join(words)
    X_input = vectorizer.transform([keyword_str])
    prediction = clf.predict(X_input)[0]
    predicted_categories.append(prediction)

In [97]:
print(classification_report(y_test, predicted_categories))

                    precision    recall  f1-score   support

           Unknown       0.00      0.00      0.00         0
          astro-ph       0.00      0.00      0.00      1344
 cond-mat.mes-hall       0.00      0.00      0.00       139
 cond-mat.mtrl-sci       0.00      0.00      0.00       169
    cond-mat.other       0.00      0.00      0.00        90
cond-mat.stat-mech       0.00      0.00      0.00       106
   cond-mat.str-el       0.00      0.00      0.00       115
     cs.IT math.IT       0.00      0.00      0.00        67
             gr-qc       0.00      0.00      0.00       161
            hep-ex       0.00      0.00      0.00       126
           hep-lat       0.00      0.00      0.00        70
            hep-ph       0.04      0.01      0.02       444
            hep-th       0.01      0.00      0.00       305
   math-ph math.MP       0.00      0.00      0.00        93
           math.AG       0.00      0.00      0.00        79
           math.CO       0.00      0.00